### What is RAG

Retrieval-Augmented Generation (RAG) is the process of optimizing the output of a large language
model, so it references an authoritative knowledge base outside of its training data sources before
generating a response. Large Language Models (LLMs) are trained on vast volumes of data and use
billions of parameters to generate original output for tasks like answering questions, translating
languages, and completing sentences. RAG extends the already powerful capabilities of LLMs to
specific domains or an organization's internal knowledge base, all without the need to retrain the
model. It is a cost-effective approach to improving LLM output so it remains relevant, accurate, and
useful in various contexts.



**Imagine ChatGPT is asked a question about One Piece but hasn't reread the manga in years. It may remember some facts but forget others. RAG lets it quickly open the relevant chapter notes before answering**



### Why RAG matters

1. Reduces Hallucinations
2. Uses Private Data
3. Keeps Knowledge Fresh(Training a model again is expensive).

### RAG Workflow

User Question ---> Convert Question → Embedding ---> Vector Search (FAISS) ---> Retrieve Relevant Chunk ---> Send Context + Question to LLM ---> Generate Grounded Answer


### Building a basic RAG Pipeline

What we will be using

1. Wikipedia article
2. Sentence transformers (for embeddings)
3. FAISS
4. Hugging Face model

(yeah that's all )

In [1]:
%pip install wikipedia sentence-transformers faiss-cpu transformers torch

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 71.9 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=887123af78a39b33f886a5cd3ceb78b6589c344b765e20cb9eaaea1ec519dc86
  Stored in directory: /root/.cache/pip/wheels/63/47/7c/a9688349aa74d228ce0a9023229c6c0ac52ca2a40fe87679b8
Successfully built wikipedia


In [2]:
import wikipedia

### Setting up the wikipedia

In [4]:
wikipedia.set_lang("en")
page=wikipedia.page("One Piece", auto_suggest=False)
text=page.content
print(text[:789])

One Piece (stylized in all caps) is a Japanese manga series written and illustrated by Eiichiro Oda. It follows the adventures of Monkey D. Luffy and his crew, the Straw Hats, as he searches for the legendary treasure known as the "One Piece" to become the next King of the Pirates. The manga has been serialized in Shueisha's shōnen manga magazine Weekly Shōnen Jump since July 1997, with its chapters compiled in 114 tankōbon volumes as of March 2026. It was licensed for an English-language release in North America and the United Kingdom by Viz Media and in Australia by Madman Entertainment.
Becoming a media franchise, it has been adapted into a festival film by Production I.G, and an anime series by Toei Animation, which premiered in October 1999. Additionally, Toei has developed


### Chunks

- Chunks = smaller text segments used in RAG.

- They are embedded, stored, and retrieved to provide relevant context to the LLM.

- Good chunking strategy = balance between size, overlap, and semantic integrity.


### Building Chunks


In [6]:
def chunk_text(text,chunk_size=500):
  chunks=[]

  for i in range(0,len(text),chunk_size):
    chunks.append(text[i:i+chunk_size])

  return chunks

chunks=chunk_text(text)
print("Chunks : ",len(chunks))

Chunks :  87


### Generating the embeddings

In [7]:
from sentence_transformers import SentenceTransformer

embedding_model= SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

chunk_embeddings= embedding_model.encode(
    chunks,
    convert_to_numpy= True,
    show_progress_bar=True
    )

print(chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

(87, 384)


87 chunks
384-dimensional vector for each chunk

### FAISS Index

- FAISS Index = vector store for embeddings in RAG.

- It enables fast similarity search to retrieve relevant chunks.

- Best for small to medium datasets; for very large corpora, managed vector DBs are preferred.

### Creating FAISS Index

In [8]:
import faiss

dimension = chunk_embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(chunk_embeddings)

print(index.ntotal)

87


In [9]:
## A simple Retrieval function

def retrieve(query, k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    distances, indices = index.search(query_embedding,k)
    return [chunks[i] for i in indices[0]]


In [11]:
retrieve("Who is Monkey D. Luffy?")

['series focuses on Monkey D. Luffy, a young man whose body has the properties of rubber after unintentionally eating the Gum-Gum Fruit—who sets off on a journey from the East Blue Sea to find the deceased King of the Pirates Gold Roger\'s ultimate treasure known as the "One Piece", and take over his prior title. Luffy sets sail as captain of the Straw Hat Pirates, and is joined by Roronoa Zoro, a swordsman and former bounty hunter; Nami, a money-obsessed thief and navigator; Usopp, a sniper and co',
 'One Piece (stylized in all caps) is a Japanese manga series written and illustrated by Eiichiro Oda. It follows the adventures of Monkey D. Luffy and his crew, the Straw Hats, as he searches for the legendary treasure known as the "One Piece" to become the next King of the Pirates. The manga has been serialized in Shueisha\'s shōnen manga magazine Weekly Shōnen Jump since July 1997, with its chapters compiled in 114 tankōbon volumes as of March 2026. It was licensed for an English-langua

### Testing Retrieval

In [12]:
query="what is haki?"
results=retrieve(query)

for i,chunk in enumerate(results):
  print(f"\n    Retrieved Chunk {i+1}    \n")
  print(chunk)


    Retrieved Chunk 1    

weakened in bodies of water, causing users to lose the ability to swim. Another supernatural power is Haki, which is an innate ability that grants its users enhanced observation and fighting abilities based on their willpower. It is one of the only effective methods of inflicting bodily harm on certain Devil Fruit users.
The surface of the Blue Planet mainly consists of the Blue Sea, two vast oceans divided by a massive continental mountain range called the Red Line, which runs along what would 

    Retrieved Chunk 2    

techniques are often mixed with other languages, and the names of several of Zoro's sword techniques are designed as jokes; they look fearsome when read by sight but sound like kinds of food when read aloud. For example, Zoro's signature move is Onigiri, which is written as demon cut but is pronounced the same as rice ball in Japanese. Eisaku Inoue, an animation director at Toei Animation, noted that these kanji readings were not used in t

### Loading a Hugging Face Model
we will be using google/flan-t5-base

In [21]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME= "google/flan-t5-base"
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [22]:
##helper function to generate answers
def generate_answer(prompt):
  inputs= tokenizer(prompt,return_tensors="pt",truncation=True)
  outputs=model.generate(**inputs,max_new_tokens=100)
  return tokenizer.decode(
      outputs[0],
      skip_special_tokens=True
  )

### Testing the model without RAG

In [32]:
question = "What is the One Piece treasure?"
answer = generate_answer(question)
print(answer)

samurai sword


### Context retrieval

In [33]:
question = "What is the One Piece treasure?"

context = "\n".join(retrieve(question))
print(context)

shared a video of Oda making final statements about the truth behind the treasure and what awaits Luffy. Oda then locked the paper inside a treasure box and sent it to the bottom of the ocean, where it will remain until the story is finished. However, multiple fans have since started planning an attempt to retrieve the treasure box early.


== Media ==


=== Manga ===

Written and illustrated by Eiichiro Oda, One Piece has been serialized by Shueisha in the shōnen manga anthology Weekly Shōnen J
One Piece (stylized in all caps) is a Japanese manga series written and illustrated by Eiichiro Oda. It follows the adventures of Monkey D. Luffy and his crew, the Straw Hats, as he searches for the legendary treasure known as the "One Piece" to become the next King of the Pirates. The manga has been serialized in Shueisha's shōnen manga magazine Weekly Shōnen Jump since July 1997, with its chapters compiled in 114 tankōbon volumes as of March 2026. It was licensed for an English-language relea

### Create a RAG Prompt

In [34]:
prompt=f"""
Use ony the context below to answer the question.
Context  : {context}
Question : {question}
Answer   :
"""

### Generating Answers With RAG

In [35]:
rag_answer=generate_answer(prompt)
print(rag_answer)

the legendary treasure


### Comparing Before RAG vs After RAG


In [36]:
print(f" QUESTION : {question}")
print()
print("WITHOUT RAG:")
print(answer)
print("WITH RAG:")
print(rag_answer)

 QUESTION : What is the One Piece treasure?

WITHOUT RAG:
samurai sword
WITH RAG:
the legendary treasure
